# Breast Cancer Diagnosis Agent — Dual Model Training

This notebook trains two models for breast cancer classification:

1. **MLP (tabular fallback)** — trained on the Wisconsin Breast Cancer dataset (30 numerical features).
2. **CNN (image primary)** — trained on the CBIS-DDSM mammography dataset.

Both models output a single sigmoid probability: 0.0 = benign, 1.0 = malignant.

In [ ]:
# ── Install dependencies ─────────────────────────────────────────────────────
# (Run this cell first; restart runtime if prompted.)

!pip install -q torch scikit-learn pandas numpy matplotlib seaborn pydicom kaggle

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset
from torchvision import transforms

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve
)

import os
import io
import zipfile
import warnings
warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print("All libraries loaded.")

---
# Part 1: MLP Training (Tabular — Wisconsin Breast Cancer Dataset)

Fallback model: used when no image is available but the user provides 30 numerical features.

In [ ]:
# ── Upload & load your CSV ────────────────────────────────────────────────────
# Option A — upload from your computer (interactive file picker):

from google.colab import files
uploaded = files.upload()                          # select your CSV file
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))

# ── Option B — load from Google Drive (uncomment to use instead) ──────────────
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv('/content/drive/MyDrive/YOUR_FOLDER/data.csv')

print(f"Loaded '{filename}': {df.shape[0]} rows, {df.shape[1]} columns")
print(df['diagnosis'].value_counts().rename({1: 'Malignant (M)', 0: 'Benign (B)'}))
df.head()

In [ ]:
# ── Preprocess ────────────────────────────────────────────────────────────────

# Drop ID column if present (non-informative)
if 'id' in df.columns:
    df = df.drop(columns=['id'])

# Drop any fully-empty trailing columns
df = df.dropna(axis=1, how='all')

# Encode target: M → 1 (malignant), B → 0 (benign)
df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0})

if df['diagnosis'].isna().any():
    raise ValueError("Unexpected values in 'diagnosis' column. Expected 'M' or 'B'.")

# Separate features and labels
FEATURE_COLS = [c for c in df.columns if c != 'diagnosis']
X = df[FEATURE_COLS].values.astype(np.float32)
y = df['diagnosis'].values.astype(np.float32)

print(f"Features: {len(FEATURE_COLS)} | Samples: {len(y)}")
print(f"   Malignant: {int(y.sum())}  |  Benign: {int((1-y).sum())}")

In [ ]:
# ── Train / Validation / Test split ───────────────────────────────────────────

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.2, random_state=SEED, stratify=y_trainval
)

# Normalise (fit only on training data)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

print(f"Split — Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}")

In [ ]:
# ── Data augmentation ─────────────────────────────────────────────────────────
# Adds copies of training samples with small Gaussian noise.

AUGMENT_FACTOR = 2     # how many extra copies to add  (0 = no augmentation)
NOISE_STD      = 0.05  # std of Gaussian noise (in normalised feature space)

def augment(X, y, factor=2, noise_std=0.05, seed=42):
    if factor == 0:
        return X, y
    rng = np.random.RandomState(seed)
    copies_X, copies_y = [X], [y]
    for _ in range(factor):
        noise = rng.normal(0, noise_std, size=X.shape).astype(np.float32)
        copies_X.append(X + noise)
        copies_y.append(y)
    return np.vstack(copies_X), np.concatenate(copies_y)

X_train_aug, y_train_aug = augment(X_train, y_train, AUGMENT_FACTOR, NOISE_STD)
print(f"Augmented training set: {len(X_train)} -> {len(X_train_aug)} samples")

In [ ]:
# ── Build PyTorch DataLoaders ─────────────────────────────────────────────────

BATCH_SIZE = 32

def to_loader(X, y, shuffle=True):
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
    return DataLoader(TensorDataset(X_t, y_t), batch_size=BATCH_SIZE, shuffle=shuffle)

train_loader = to_loader(X_train_aug, y_train_aug, shuffle=True)
val_loader   = to_loader(X_val, y_val, shuffle=False)
test_loader  = to_loader(X_test, y_test, shuffle=False)

In [ ]:
# ── Define the MLP model ──────────────────────────────────────────────────────

class BreastCancerMLP(nn.Module):
    """
    Multi-layer perceptron for breast cancer classification.
    Input:  30 normalised tumour features.
    Output: single sigmoid value in [0, 1].
    """
    def __init__(self, input_dim=30):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

mlp_model = BreastCancerMLP(input_dim=X_train.shape[1]).to(device)
print(mlp_model)

In [ ]:
# ── Train the MLP ─────────────────────────────────────────────────────────────

EPOCHS        = 100
LEARNING_RATE = 1e-3
PATIENCE      = 15

criterion = nn.BCELoss()
optimizer = optim.Adam(mlp_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

best_val_loss = float('inf')
best_weights  = None
patience_ctr  = 0
history       = {'train_loss': [], 'val_loss': [], 'val_auc': []}

for epoch in range(1, EPOCHS + 1):
    mlp_model.train()
    train_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        preds = mlp_model(X_batch)
        loss  = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(X_batch)
    train_loss /= len(train_loader.dataset)

    mlp_model.eval()
    val_loss, val_preds, val_true = 0.0, [], []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            preds    = mlp_model(X_batch)
            val_loss += criterion(preds, y_batch).item() * len(X_batch)
            val_preds.extend(preds.cpu().numpy().flatten())
            val_true.extend(y_batch.cpu().numpy().flatten())
    val_loss /= len(val_loader.dataset)
    val_auc   = roc_auc_score(val_true, val_preds)

    scheduler.step(val_loss)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_auc'].append(val_auc)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_weights  = {k: v.clone() for k, v in mlp_model.state_dict().items()}
        patience_ctr  = 0
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f"Early stop at epoch {epoch}  (best val loss: {best_val_loss:.4f})")
            break

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d} | train_loss: {train_loss:.4f} | "
              f"val_loss: {val_loss:.4f} | val_AUC: {val_auc:.4f}")

mlp_model.load_state_dict(best_weights)
print("\nMLP training complete. Best weights restored.")

In [ ]:
# ── Plot MLP training curves ──────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history['train_loss'], label='Train loss')
axes[0].plot(history['val_loss'],   label='Val loss')
axes[0].set_title('Loss over epochs')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history['val_auc'], color='green', label='Val AUC-ROC')
axes[1].set_title('Validation AUC-ROC over epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylim(0.5, 1.0)
axes[1].legend()

plt.tight_layout()
plt.savefig('mlp_training_curves.png', dpi=150)
plt.show()

In [ ]:
# ── Evaluate MLP on test set ──────────────────────────────────────────────────

mlp_model.eval()
test_preds_prob, test_true = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        prob = mlp_model(X_batch).cpu().numpy().flatten()
        test_preds_prob.extend(prob)
        test_true.extend(y_batch.numpy().flatten())

test_preds_prob = np.array(test_preds_prob)
test_true       = np.array(test_true)
test_preds_bin  = (test_preds_prob >= 0.5).astype(int)

acc = accuracy_score(test_true, test_preds_bin)
auc = roc_auc_score(test_true, test_preds_prob)

print("=" * 45)
print(f"  Test Accuracy : {acc*100:.2f}%")
print(f"  Test AUC-ROC  : {auc:.4f}")
print("=" * 45)
print(classification_report(test_true, test_preds_bin,
                             target_names=['Benign (B)', 'Malignant (M)']))

# Confusion matrix
cm = confusion_matrix(test_true, test_preds_bin)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred: Benign', 'Pred: Malignant'],
            yticklabels=['True: Benign', 'True: Malignant'])
plt.title('MLP Confusion Matrix (Test Set)')
plt.tight_layout()
plt.savefig('mlp_confusion_matrix.png', dpi=150)
plt.show()

# ROC curve
fpr, tpr, _ = roc_curve(test_true, test_preds_prob)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='crimson', lw=2, label=f'ROC curve (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Breast Cancer MLP')
plt.legend()
plt.tight_layout()
plt.savefig('mlp_roc_curve.png', dpi=150)
plt.show()

In [ ]:
# ── Save MLP weights ──────────────────────────────────────────────────────────
# The MLP expects raw (unnormalised) features — the inference code in
# models/breast_cancer.py handles normalisation at runtime.
# However, the notebook trains on normalised data, so we save a note
# that the saved weights assume StandardScaler was applied.

import pickle

torch.save(mlp_model.state_dict(), 'breast_cancer_mlp.pt')

with open('breast_cancer_mlp_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Saved: breast_cancer_mlp.pt")
print("Saved: breast_cancer_mlp_scaler.pkl")
print()
print("NOTE: The inference code in models/breast_cancer.py does NOT apply the")
print("StandardScaler at runtime. The raw feature values from the user are fed")
print("directly into the MLP. To make this work correctly, either:")
print("  (a) Retrain the MLP on unscaled data (remove the StandardScaler step), or")
print("  (b) Update _preprocess_features() in models/breast_cancer.py to apply")
print("      the saved scaler before feeding features into the model.")
print()
print("For now, we retrain the MLP on raw (unscaled) data so it matches")
print("the inference pipeline.")

In [ ]:
# ── Retrain MLP on raw (unscaled) data ────────────────────────────────────────
# The inference code in models/breast_cancer.py feeds raw feature values
# directly into the MLP, so we need to train on unscaled data.

X_trainval_raw, X_test_raw, y_trainval_raw, y_test_raw = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
X_train_raw, X_val_raw, y_train_raw, y_val_raw = train_test_split(
    X_trainval_raw, y_trainval_raw, test_size=0.2, random_state=SEED, stratify=y_trainval_raw
)

# Augment
X_train_raw_aug, y_train_raw_aug = augment(X_train_raw, y_train_raw, AUGMENT_FACTOR, NOISE_STD)

def to_loader_raw(X, y, shuffle=True):
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
    return DataLoader(TensorDataset(X_t, y_t), batch_size=BATCH_SIZE, shuffle=shuffle)

train_loader_raw = to_loader_raw(X_train_raw_aug, y_train_raw_aug, shuffle=True)
val_loader_raw   = to_loader_raw(X_val_raw, y_val_raw, shuffle=False)
test_loader_raw  = to_loader_raw(X_test_raw, y_test_raw, shuffle=False)

# Train a fresh MLP on raw data
mlp_raw = BreastCancerMLP(input_dim=X_train_raw.shape[1]).to(device)

criterion_raw = nn.BCELoss()
optimizer_raw = optim.Adam(mlp_raw.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler_raw = optim.lr_scheduler.ReduceLROnPlateau(optimizer_raw, patience=5, factor=0.5)

best_val_loss_raw = float('inf')
best_weights_raw  = None
patience_ctr_raw  = 0

for epoch in range(1, EPOCHS + 1):
    mlp_raw.train()
    train_loss = 0.0
    for X_batch, y_batch in train_loader_raw:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer_raw.zero_grad()
        preds = mlp_raw(X_batch)
        loss  = criterion_raw(preds, y_batch)
        loss.backward()
        optimizer_raw.step()
        train_loss += loss.item() * len(X_batch)
    train_loss /= len(train_loader_raw.dataset)

    mlp_raw.eval()
    val_loss, val_preds, val_true = 0.0, [], []
    with torch.no_grad():
        for X_batch, y_batch in val_loader_raw:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            preds    = mlp_raw(X_batch)
            val_loss += criterion_raw(preds, y_batch).item() * len(X_batch)
            val_preds.extend(preds.cpu().numpy().flatten())
            val_true.extend(y_batch.cpu().numpy().flatten())
    val_loss /= len(val_loader_raw.dataset)
    val_auc   = roc_auc_score(val_true, val_preds)

    scheduler_raw.step(val_loss)

    if val_loss < best_val_loss_raw:
        best_val_loss_raw = val_loss
        best_weights_raw  = {k: v.clone() for k, v in mlp_raw.state_dict().items()}
        patience_ctr_raw  = 0
    else:
        patience_ctr_raw += 1
        if patience_ctr_raw >= PATIENCE:
            print(f"Early stop at epoch {epoch}  (best val loss: {best_val_loss_raw:.4f})")
            break

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d} | train_loss: {train_loss:.4f} | "
              f"val_loss: {val_loss:.4f} | val_AUC: {val_auc:.4f}")

mlp_raw.load_state_dict(best_weights_raw)

# Evaluate on raw test data
mlp_raw.eval()
test_preds_prob_raw, test_true_raw = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader_raw:
        X_batch = X_batch.to(device)
        prob = mlp_raw(X_batch).cpu().numpy().flatten()
        test_preds_prob_raw.extend(prob)
        test_true_raw.extend(y_batch.numpy().flatten())

test_preds_prob_raw = np.array(test_preds_prob_raw)
test_preds_bin_raw  = (test_preds_prob_raw >= 0.5).astype(int)

acc_raw = accuracy_score(test_true_raw, test_preds_bin_raw)
auc_raw = roc_auc_score(test_true_raw, test_preds_prob_raw)

print("\n" + "=" * 45)
print(f"  Raw MLP Test Accuracy : {acc_raw*100:.2f}%")
print(f"  Raw MLP Test AUC-ROC  : {auc_raw:.4f}")
print("=" * 45)

# Save the raw-trained MLP
torch.save(mlp_raw.state_dict(), 'breast_cancer_mlp.pt')
print("\nSaved: breast_cancer_mlp.pt (trained on raw/unscaled features)")

---
# Part 2: CNN Training (Image — CBIS-DDSM Mammography Dataset)

Primary model: used when a breast scan image is provided.

In [ ]:
# ── Download CBIS-DDSM from Kaggle ────────────────────────────────────────────
# You need a Kaggle API key. Get one from: https://www.kaggle.com/settings
# Then upload kaggle.json or set KAGGLE_USERNAME and KAGGLE_KEY env vars.

# Upload kaggle.json
from google.colab import files
print("Upload your kaggle.json file (from kaggle.com/settings):")
uploaded = files.upload()
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'wb') as f:
    f.write(uploaded['kaggle.json'])
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

# Download and extract the dataset
!kaggle datasets download -d awsaf49/cbis-ddsm-breast-cancer-dataset -p /content/cbis-ddsm
!unzip -q /content/cbis-ddsm/cbis-ddsm-breast-cancer-dataset.zip -d /content/cbis-ddsm/data

print("Dataset downloaded.")

In [ ]:
# ── Load and prepare CBIS-DDSM metadata ───────────────────────────────────────

# CBIS-DDSM provides a CSV with image paths and labels
# Labels: M = malignant, B = benign

DATA_DIR = '/content/cbis-ddsm/data'

# Find the CSV file (path may vary based on dataset version)
csv_candidates = [
    os.path.join(DATA_DIR, 'csv/dicom_info.csv'),
    os.path.join(DATA_DIR, 'dicom_info.csv'),
]

csv_path = None
for candidate in csv_candidates:
    if os.path.exists(candidate):
        csv_path = candidate
        break

if csv_path is None:
    # Try to find any CSV in the data directory
    for root, dirs, files in os.walk(DATA_DIR):
        for f in files:
            if f.endswith('.csv'):
                csv_path = os.path.join(root, f)
                break
        if csv_path:
            break

if csv_path is None:
    raise FileNotFoundError(f"No CSV found in {DATA_DIR}. Check dataset structure.")

print(f"Using CSV: {csv_path}")
dicom_df = pd.read_csv(csv_path)
print(f"Columns: {list(dicom_df.columns)}")
print(f"Shape: {dicom_df.shape}")
dicom_df.head()

In [ ]:
# ── Filter for full mammogram images and map labels ───────────────────────────

# The CBIS-DDSM dataset typically has columns like:
#   'filename', 'label', 'patient_id', etc.
# Or it may use 'image_path' and 'breast_density' / 'pathology'
# We need to adapt to the actual column names.

# Print unique labels to understand the data
label_col = None
for col in ['label', 'pathology', 'diagnosis', 'class']:
    if col in dicom_df.columns:
        label_col = col
        break

if label_col is None:
    print("Available columns:", list(dicom_df.columns))
    print("\nPlease identify the label column and update the code below.")
else:
    print(f"Label column: '{label_col}'")
    print(f"Unique values: {dicom_df[label_col].unique()}")
    print(f"Value counts:\n{dicom_df[label_col].value_counts()}")

In [ ]:
# ── Build image paths and binary labels ───────────────────────────────────────

# Adapt this cell based on the actual CSV structure.
# The goal: create a DataFrame with columns ['image_path', 'label']
# where label is 0 (benign) or 1 (malignant).

# Example mapping (adjust based on actual label values):
# 'BENIGN' -> 0, 'MALIGNANT' -> 1, 'BENIGN_WITHOUT_CALLBACK' -> 0

def map_label(val):
    val = str(val).upper().strip()
    if 'MALIGNANT' in val:
        return 1
    elif 'BENIGN' in val:
        return 0
    else:
        return None

dicom_df['binary_label'] = dicom_df[label_col].apply(map_label)

# Drop rows with unmapped labels
dicom_df = dicom_df.dropna(subset=['binary_label'])
dicom_df['binary_label'] = dicom_df['binary_label'].astype(int)

# Find the image path column
img_path_col = None
for col in ['image_path', 'file_path', 'filename', 'path', 'SOP Instance UID']:
    if col in dicom_df.columns:
        img_path_col = col
        break

if img_path_col is None:
    print("Available columns:", list(dicom_df.columns))
    raise ValueError("Cannot find image path column. Please identify it manually.")

print(f"Image path column: '{img_path_col}'")
print(f"Samples: {len(dicom_df)} | Benign: {(dicom_df['binary_label']==0).sum()} | Malignant: {(dicom_df['binary_label']==1).sum()}")

In [ ]:
# ── Dataset and transforms ────────────────────────────────────────────────────

IMG_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class CBISDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]

        # Load DICOM or standard image
        try:
            if img_path.endswith('.dcm') or img_path.endswith('.dicom'):
                import pydicom
                ds = pydicom.dcmread(img_path)
                img = ds.pixel_array
                img = Image.fromarray(img).convert('RGB')
            else:
                img = Image.open(img_path).convert('RGB')
        except Exception as e:
            # Return a black image on failure
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (0, 0, 0))

        if self.transform:
            img = self.transform(img)

        return img, torch.tensor(label, dtype=torch.float32)


print("Dataset class defined.")

In [ ]:
# ── Split data and create DataLoaders ─────────────────────────────────────────

all_paths = dicom_df[img_path_col].tolist()
all_labels = dicom_df['binary_label'].tolist()

# Split: 70% train, 15% val, 15% test
train_paths, test_paths, train_labels, test_labels = train_test_split(
    all_paths, all_labels, test_size=0.15, random_state=SEED, stratify=all_labels
)
train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_paths, train_labels, test_size=0.176, random_state=SEED, stratify=train_labels  # 0.176 * 0.85 ≈ 0.15
)

train_dataset = CBISDataset(train_paths, train_labels, transform=train_transform)
val_dataset   = CBISDataset(val_paths, val_labels, transform=val_transform)
test_dataset  = CBISDataset(test_paths, test_labels, transform=val_transform)

BATCH_SIZE_CNN = 32

train_loader_cnn = DataLoader(train_dataset, batch_size=BATCH_SIZE_CNN, shuffle=True, num_workers=2)
val_loader_cnn   = DataLoader(val_dataset, batch_size=BATCH_SIZE_CNN, shuffle=False, num_workers=2)
test_loader_cnn  = DataLoader(test_dataset, batch_size=BATCH_SIZE_CNN, shuffle=False, num_workers=2)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

In [ ]:
# ── Define the CNN model ──────────────────────────────────────────────────────

class BreastCancerCNN(nn.Module):
    """
    CNN for breast cancer classification (benign vs malignant).
    Input: RGB breast scan image (224x224).
    Output: single sigmoid probability in [0, 1].
    """
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


cnn_model = BreastCancerCNN().to(device)
print(f"Using device: {device}")
print(cnn_model)

In [ ]:
# ── Train the CNN ─────────────────────────────────────────────────────────────

EPOCHS_CNN        = 50
LEARNING_RATE_CNN = 1e-4
PATIENCE_CNN      = 10

criterion_cnn = nn.BCELoss()
optimizer_cnn = optim.Adam(cnn_model.parameters(), lr=LEARNING_RATE_CNN, weight_decay=1e-4)
scheduler_cnn = optim.lr_scheduler.ReduceLROnPlateau(optimizer_cnn, patience=5, factor=0.5)

best_val_loss_cnn = float('inf')
best_weights_cnn  = None
patience_ctr_cnn  = 0
history_cnn       = {'train_loss': [], 'val_loss': [], 'val_auc': []}

for epoch in range(1, EPOCHS_CNN + 1):
    cnn_model.train()
    train_loss = 0.0
    for images, labels in train_loader_cnn:
        images, labels = images.to(device), labels.to(device).unsqueeze(1)
        optimizer_cnn.zero_grad()
        preds = cnn_model(images)
        loss  = criterion_cnn(preds, labels)
        loss.backward()
        optimizer_cnn.step()
        train_loss += loss.item() * len(images)
    train_loss /= len(train_loader_cnn.dataset)

    cnn_model.eval()
    val_loss, val_preds, val_true = 0.0, [], []
    with torch.no_grad():
        for images, labels in val_loader_cnn:
            images, labels = images.to(device), labels.to(device).unsqueeze(1)
            preds    = cnn_model(images)
            val_loss += criterion_cnn(preds, labels).item() * len(images)
            val_preds.extend(preds.cpu().numpy().flatten())
            val_true.extend(labels.cpu().numpy().flatten())
    val_loss /= len(val_loader_cnn.dataset)
    val_auc   = roc_auc_score(val_true, val_preds)

    scheduler_cnn.step(val_loss)
    history_cnn['train_loss'].append(train_loss)
    history_cnn['val_loss'].append(val_loss)
    history_cnn['val_auc'].append(val_auc)

    if val_loss < best_val_loss_cnn:
        best_val_loss_cnn = val_loss
        best_weights_cnn  = {k: v.clone() for k, v in cnn_model.state_dict().items()}
        patience_ctr_cnn  = 0
    else:
        patience_ctr_cnn += 1
        if patience_ctr_cnn >= PATIENCE_CNN:
            print(f"Early stop at epoch {epoch}  (best val loss: {best_val_loss_cnn:.4f})")
            break

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d} | train_loss: {train_loss:.4f} | "
              f"val_loss: {val_loss:.4f} | val_AUC: {val_auc:.4f}")

cnn_model.load_state_dict(best_weights_cnn)
print("\nCNN training complete. Best weights restored.")

In [ ]:
# ── Plot CNN training curves ──────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history_cnn['train_loss'], label='Train loss')
axes[0].plot(history_cnn['val_loss'],   label='Val loss')
axes[0].set_title('CNN Loss over epochs')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history_cnn['val_auc'], color='green', label='Val AUC-ROC')
axes[1].set_title('CNN Validation AUC-ROC over epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylim(0.5, 1.0)
axes[1].legend()

plt.tight_layout()
plt.savefig('cnn_training_curves.png', dpi=150)
plt.show()

In [ ]:
# ── Evaluate CNN on test set ──────────────────────────────────────────────────

cnn_model.eval()
test_preds_prob_cnn, test_true_cnn = [], []
with torch.no_grad():
    for images, labels in test_loader_cnn:
        images = images.to(device)
        prob = cnn_model(images).cpu().numpy().flatten()
        test_preds_prob_cnn.extend(prob)
        test_true_cnn.extend(labels.numpy().flatten())

test_preds_prob_cnn = np.array(test_preds_prob_cnn)
test_true_cnn       = np.array(test_true_cnn)
test_preds_bin_cnn  = (test_preds_prob_cnn >= 0.5).astype(int)

acc_cnn = accuracy_score(test_true_cnn, test_preds_bin_cnn)
auc_cnn = roc_auc_score(test_true_cnn, test_preds_prob_cnn)

print("=" * 45)
print(f"  CNN Test Accuracy : {acc_cnn*100:.2f}%")
print(f"  CNN Test AUC-ROC  : {auc_cnn:.4f}")
print("=" * 45)
print(classification_report(test_true_cnn, test_preds_bin_cnn,
                             target_names=['Benign (B)', 'Malignant (M)']))

# Confusion matrix
cm_cnn = confusion_matrix(test_true_cnn, test_preds_bin_cnn)
plt.figure(figsize=(5, 4))
sns.heatmap(cm_cnn, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred: Benign', 'Pred: Malignant'],
            yticklabels=['True: Benign', 'True: Malignant'])
plt.title('CNN Confusion Matrix (Test Set)')
plt.tight_layout()
plt.savefig('cnn_confusion_matrix.png', dpi=150)
plt.show()

# ROC curve
fpr_cnn, tpr_cnn, _ = roc_curve(test_true_cnn, test_preds_prob_cnn)
plt.figure(figsize=(6, 5))
plt.plot(fpr_cnn, tpr_cnn, color='crimson', lw=2, label=f'ROC curve (AUC = {auc_cnn:.3f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Breast Cancer CNN')
plt.legend()
plt.tight_layout()
plt.savefig('cnn_roc_curve.png', dpi=150)
plt.show()

In [ ]:
# ── Sample predictions ────────────────────────────────────────────────────────

cnn_model.eval()
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, ax in enumerate(axes.flat):
    if i >= len(test_dataset):
        break
    img, label = test_dataset[i]
    img_batch = img.unsqueeze(0).to(device)
    with torch.no_grad():
        prob = cnn_model(img_batch).item()
    pred_label = 'Malignant' if prob >= 0.5 else 'Benign'
    true_label = 'Malignant' if label.item() == 1 else 'Benign'

    # Unnormalise for display
    img_display = img.cpu().numpy().transpose(1, 2, 0)
    img_display = img_display * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    img_display = np.clip(img_display, 0, 1)

    ax.imshow(img_display)
    color = 'green' if pred_label == true_label else 'red'
    ax.set_title(f"Pred: {pred_label} ({prob:.2f})\nTrue: {true_label}", fontsize=10, color=color)
    ax.axis('off')

plt.tight_layout()
plt.savefig('cnn_sample_predictions.png', dpi=150)
plt.show()

In [ ]:
# ── Save CNN weights ──────────────────────────────────────────────────────────

torch.save(cnn_model.state_dict(), 'breast_cancer_cnn.pt')
print("Saved: breast_cancer_cnn.pt")

# Copy both models to Google Drive for persistence
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy('breast_cancer_mlp.pt', '/content/drive/MyDrive/YOUR_FOLDER/breast_cancer_mlp.pt')
# shutil.copy('breast_cancer_cnn.pt', '/content/drive/MyDrive/YOUR_FOLDER/breast_cancer_cnn.pt')
# print("Files copied to Google Drive.")

In [ ]:
# ── Verify both models load correctly ─────────────────────────────────────────

# Test MLP
mlp_verify = BreastCancerMLP(input_dim=30).to(device)
mlp_verify.load_state_dict(torch.load('breast_cancer_mlp.pt', map_location=device))
mlp_verify.eval()
dummy_tabular = torch.randn(1, 30).to(device)
with torch.no_grad():
    mlp_out = mlp_verify(dummy_tabular).item()
print(f"MLP loaded OK — dummy output: {mlp_out:.4f}")

# Test CNN
cnn_verify = BreastCancerCNN().to(device)
cnn_verify.load_state_dict(torch.load('breast_cancer_cnn.pt', map_location=device))
cnn_verify.eval()
dummy_image = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
with torch.no_grad():
    cnn_out = cnn_verify(dummy_image).item()
print(f"CNN loaded OK — dummy output: {cnn_out:.4f}")

print("\nBoth models verified. Copy breast_cancer_mlp.pt and breast_cancer_cnn.pt")
print("to saved_models/ in the project root.")